In [ ]:
cd ../..

In [ ]:
import csv
import os

from IPython.display import display, HTML
from diff_match_patch import diff_match_patch
import pandas as pd
from scripts.globals import PYENSEMBL_CACHE_DIR
from pyensembl import EnsemblRelease

# pyensembl
os.environ["PYENSEMBL_CACHE_DIR"] = PYENSEMBL_CACHE_DIR
ens60 = EnsemblRelease(60)
ens60.download()
ens60.index()

# reading CLASH txt and saving as csv

In [ ]:
with open("data/clash/mmc1.txt") as f:
    lines = iter(f)
    data = []
    for line in lines:
        if line.startswith("#"):
            line = line[1:]
            row = next(csv.reader([line], delimiter="\t"))
        else:
            row = next(csv.reader([line], delimiter="\t"))
            data.append(row)

    with open("data/clash/clash_raw.csv", "w") as f:
        writer = csv.writer(f, delimiter=",")
        writer.writerows(data)

# wrangling values

In [ ]:
df = pd.read_csv("data/clash/clash_raw.csv")

In [ ]:
# dropping unused CLASH columns
cols_to_drop = ["miRNA_start", "miRNA_end", "chimeras_decompressed",
                "experiments", "experiments_list", "microRNA_first", "two_way_merged",
                "log2_target_enrichment", "CLASH_single_reads_ovlp",
                "5'UTR", "CDS", "3'UTR", "conservation_score",
                "CLASH_cluster_ovlp", "PAR_CLIP_cluster_ovlp"]

df.drop(columns=cols_to_drop, inplace=True)

In [ ]:
# process microRNA_name and mRNA_name columns
new_mirna_cols = df["microRNA_name"].str.split("_", expand=True)
new_mirna_cols.columns = ["mirna_accession", "temp1", "mirna_name", "temp2"]
df = pd.concat([df, new_mirna_cols], axis=1)

new_mrna_cols = df["mRNA_name"].str.split("_", expand=True)
new_mrna_cols.columns = ["ensg", "enst", "gene_name", "temp3"]
df = pd.concat([df, new_mrna_cols], axis=1)

# dropping temporary columns
temp_cols = ["microRNA_name", "mRNA_name", "temp1", "temp2", "temp3"]
df.drop(columns=temp_cols, inplace=True)

In [ ]:
# converting start:end biological coordinates into 0-based index
# subtracting 1 from start is enough.

df["mRNA_start"] = df["mRNA_start"] - 1
df["mRNA_end_extended"] = df["mRNA_end_extended"]

In [ ]:
# renaming columns
rename_dict = {
    "seq_ID": "id",
    "miRNA_seq": "mirna_sequence",
    "mRNA_seq_extended": "mrna_sequence",
    "mRNA_start": "true_start_index",
    "mRNA_end_extended": "true_end_index",
    "seed_type": "true_seed_type",
    "folding_class": "true_folding_class"
}

df = df.rename(columns=rename_dict)

# augmenting df with full sequences of transcripts using pyensembl

In [ ]:
# get sequences of ENSTs from ENSEMBL 60 to a dict
seq_dict = {
    i: ens60.transcript_by_id(i).sequence
    if ens60.transcript_by_id(i).sequence
    else None
    for i in df.enst.unique().tolist()
}

# appending full sequences
df["full_sequence_of_transcript"] = df["enst"].map(seq_dict)

In [ ]:
# getting sequence slices from start:end positions
# if slice is closer to the 5' end and can't be extended for 30 nucleotides,
# then return the start position as 0

def get_sequence_slice(row):
    sequence = row["full_sequence_of_transcript"]
    start = max(row["true_start_index"], 0)
    end = row["true_end_index"]
    return sequence[start:end], start, end


df[["extended_sequence", 'extended_start', "extended_end"]] = df.apply(
    lambda row: get_sequence_slice(row), axis=1, result_type='expand')


df.head()

# checking differences between fetched & df sequences

In [ ]:
def compare_sequences(df):
    df["substring"] = df.apply(lambda row: row["full_sequence_of_transcript"]
                               [row["true_start_index"]:row["true_end_index"]], axis=1)
    df["comparison_result"] = df["substring"] == df["mrna_sequence"]
    return df


print(compare_sequences(df).comparison_result.value_counts())
print("6 rows' sequences are different")

In [ ]:
compare_sequences(df)[compare_sequences(
    df)["comparison_result"] == False].gene_name.unique()

In [ ]:
def highlight_differences(string1, string2):
    dmp = diff_match_patch()
    diffs = dmp.diff_main(string1, string2)
    dmp.diff_cleanupSemantic(diffs)

    highlighted_string = ""
    for op, data in diffs:
        if op == 0:  # no change
            highlighted_string += data
        elif op == -1:  # deletion
            highlighted_string += f'<del>{data}</del>'
        elif op == 1:  # insertion
            highlighted_string += f'<ins>{data}</ins>'

    return highlighted_string


# Assuming you have a DataFrame called 'df' with a 'highlighted' column
def add_color(row):
    highlighted = row['highlighted']
    if '<del>' in highlighted:
        highlighted = highlighted.replace(
            '<del>', '<span style="color: red;">')
        highlighted = highlighted.replace('</del>', '</span>')
    if '<ins>' in highlighted:
        highlighted = highlighted.replace(
            '<ins>', '<span style="color: green;">')
        highlighted = highlighted.replace('</ins>', '</span>')
    return highlighted


different_values_df = compare_sequences(
    df)[compare_sequences(df)["comparison_result"] == False]

# creating difference strings
different_values_df['highlighted'] = different_values_df.apply(
    lambda row: highlight_differences(row['mrna_sequence'], row['extended_sequence']), axis=1)


# render the differences
different_values_df['highlighted'] = different_values_df.apply(
    add_color, axis=1)

display(HTML(different_values_df[["highlighted"]].to_html(escape=False)))

print("reds symbolize sequences deleted from df sequences")
print("greens symbolize sequences added by the ENSEMBL60 sequence fetch tool")
print("whites symbolize sequences that are the same")

# extending both end of sequence for 30 nts 

In [ ]:
def get_sequence_slice(row):
    sequence = row["full_sequence_of_transcript"]
    start = max(row["true_start_index"]-30, 0)
    end = row["true_end_index"]+30
    return sequence[start:end], start, end


df[["extended_sequence", 'extended_start', "extended_end"]] = df.apply(
    lambda row: get_sequence_slice(row), axis=1, result_type='expand')


df.head()

In [ ]:
# dropping temp cols and saving to csv

cols_to_drop = ["substring", "comparison_result"]

df.drop(columns=cols_to_drop, inplace=True)
df.head()

In [ ]:
df.to_csv("data/clash/clash_parsed.csv", index=False)